# 004 Human-in-the-loop

这是 LangChain Advanced usage 学习线的第四份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/human-in-the-loop

学习目标：

1. 理解 human-in-the-loop 解决的是高风险动作审批问题
2. 学会使用 `HumanInTheLoopMiddleware`
3. 理解为什么 HITL 必须配合 checkpointer
4. 跑通 approve / reject / edit 三种恢复决策
5. 理解 thread_id 如何让暂停任务可恢复
6. 对比本仓库 Harness 的 `approval_required` 和 `stream_resume_approval`

这一讲使用 fake model，但中断和恢复机制是真实的 LangGraph 执行流程。工具只写入内存列表，不写文件。

## 1. 为什么需要 HITL

模型生成 tool call 不代表工具应该立刻执行。

这些动作通常需要人工审批：

- 写文件
- 执行 shell
- 发邮件
- 修改数据库
- 调用第三方下单 / 支付 / 删除接口

HITL 的核心流程：

```text
模型提出工具调用
  -> middleware 中断
  -> 返回 action_requests 给应用层
  -> 用户审批
  -> Command(resume=...) 恢复任务
  -> 工具才真正执行
```

这和本仓库 Harness 的 approval 流程是同一类设计。

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


SIDE_EFFECT_LOG: list[dict] = []


@tool
def write_note(path: str, content: str) -> str:
    """Write a note into an in-memory log."""
    SIDE_EFFECT_LOG.append({"path": path, "content": content})
    print("WRITE EXECUTED:", path, content)
    return "written " + path


def make_hitl_agent(final_answer: str = "操作已完成。"):
    model = ToolCallingFakeModel(
        responses=[
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "write_note",
                        "args": {"path": "demo.txt", "content": "hello"},
                        "id": "call_1",
                    }
                ],
            ),
            AIMessage(content=final_answer),
        ]
    )

    return create_agent(
        model=model,
        tools=[write_note],
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"write_note": True})],
        checkpointer=InMemorySaver(),
    )


def print_interrupts(result) -> None:
    print("interrupt_count:", len(result.interrupts))
    for interrupt in result.interrupts:
        print("interrupt_id:", interrupt.id)
        print(interrupt.value)


def print_messages(result) -> None:
    for message in result.value.get("messages", []):
        print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)


## 2. 第一次调用：任务中断，工具不会执行

HITL 必须配合 checkpointer，因为任务会暂停。

这里使用 `InMemorySaver`。生产环境应该换成持久化 checkpointer。

In [29]:
SIDE_EFFECT_LOG.clear()

approve_agent = make_hitl_agent()
approve_config = {"configurable": {"thread_id": "hitl-approve-demo"}}

initial_result = approve_agent.invoke(
    {"messages": [{"role": "user", "content": "请写一条 note"}]},
    config=approve_config,
    version="v2",
)

print_interrupts(initial_result)
print("side effects before approval:", SIDE_EFFECT_LOG)


interrupt_count: 1
interrupt_id: b8f5b9f2795e1b18d826649382db6465
{'action_requests': [{'name': 'write_note', 'args': {'path': 'demo.txt', 'content': 'hello'}, 'description': "Tool execution requires approval\n\nTool: write_note\nArgs: {'path': 'demo.txt', 'content': 'hello'}"}], 'review_configs': [{'action_name': 'write_note', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}
side effects before approval: []


观察点：

- 返回的是 `GraphOutput`
- `interrupts` 里有 `action_requests`
- `review_configs` 告诉前端允许哪些决策
- 工具还没有执行，所以 `SIDE_EFFECT_LOG` 为空

## 3. Approve：批准后继续执行工具

批准时用：

```python
Command(resume={"decisions": [{"type": "approve"}]})
```

注意必须使用同一个 `thread_id`，否则系统不知道要恢复哪一个暂停任务。

In [30]:
approved_result = approve_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=approve_config,
    version="v2",
)

print_messages(approved_result)
print("side effects after approval:", SIDE_EFFECT_LOG)


WRITE EXECUTED: demo.txt hello
human 请写一条 note
ai 
tool_calls: [{'name': 'write_note', 'args': {'path': 'demo.txt', 'content': 'hello'}, 'id': 'call_1', 'type': 'tool_call'}]
tool written demo.txt
ai 操作已完成。
side effects after approval: [{'path': 'demo.txt', 'content': 'hello'}]


## 4. Reject：拒绝后工具不执行

拒绝时可以给出原因：

```python
{"type": "reject", "message": "不允许写文件"}
```

拒绝后会生成一个 tool message，但真实工具不会执行。

In [31]:
SIDE_EFFECT_LOG.clear()

reject_agent = make_hitl_agent(final_answer="已取消写入。")
reject_config = {"configurable": {"thread_id": "hitl-reject-demo"}}

reject_initial = reject_agent.invoke(
    {"messages": [{"role": "user", "content": "请写一条 note"}]},
    config=reject_config,
    version="v2",
)

print_interrupts(reject_initial)

rejected_result = reject_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "不允许写入 demo.txt"}]}),
    config=reject_config,
    version="v2",
)

print_messages(rejected_result)
print("side effects after rejection:", SIDE_EFFECT_LOG)


interrupt_count: 1
interrupt_id: 5e42c60d06184cf031c0781c1db59d37
{'action_requests': [{'name': 'write_note', 'args': {'path': 'demo.txt', 'content': 'hello'}, 'description': "Tool execution requires approval\n\nTool: write_note\nArgs: {'path': 'demo.txt', 'content': 'hello'}"}], 'review_configs': [{'action_name': 'write_note', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}
human 请写一条 note
ai 
tool_calls: [{'name': 'write_note', 'args': {'path': 'demo.txt', 'content': 'hello'}, 'id': 'call_1', 'type': 'tool_call'}]
tool 不允许写入 demo.txt
ai 已取消写入。
side effects after rejection: []


## 5. Edit：审批时修改工具参数

有时不是完全拒绝，而是要改参数。

比如模型想写 `demo.txt`，人类审批时改成 `safe.txt`。

`edit` 的结构是：

```python
{
    "type": "edit",
    "edited_action": {
        "name": "write_note",
        "args": {"path": "safe.txt", "content": "edited"},
    },
}
```

In [32]:
SIDE_EFFECT_LOG.clear()

edit_agent = make_hitl_agent(final_answer="已按修改后的参数写入。")
edit_config = {"configurable": {"thread_id": "hitl-edit-demo"}}

edit_initial = edit_agent.invoke(
    {"messages": [{"role": "user", "content": "请写一条 note"}]},
    config=edit_config,
    version="v2",
)

print_interrupts(edit_initial)

edited_result = edit_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "write_note",
                        "args": {"path": "safe.txt", "content": "edited content"},
                    },
                }
            ]
        }
    ),
    config=edit_config,
    version="v2",
)

print_messages(edited_result)
print("side effects after edit:", SIDE_EFFECT_LOG)


interrupt_count: 1
interrupt_id: bc5860e9c2a8c6686cf00521e81d5010
{'action_requests': [{'name': 'write_note', 'args': {'path': 'demo.txt', 'content': 'hello'}, 'description': "Tool execution requires approval\n\nTool: write_note\nArgs: {'path': 'demo.txt', 'content': 'hello'}"}], 'review_configs': [{'action_name': 'write_note', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}
WRITE EXECUTED: safe.txt edited content
human 请写一条 note
ai 
tool_calls: [{'type': 'tool_call', 'name': 'write_note', 'args': {'path': 'safe.txt', 'content': 'edited content'}, 'id': 'call_1'}]
tool written safe.txt
ai 已按修改后的参数写入。
side effects after edit: [{'path': 'safe.txt', 'content': 'edited content'}]


## 6. 限制允许的决策

`interrupt_on` 不只能写 `True`，也可以为每个工具指定允许的决策。

例如只允许 approve / reject，不允许 edit：

In [33]:
restricted_hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "write_note": {
            "allowed_decisions": ["approve", "reject"],
            "description": "write_note 会产生写入副作用，需要审批。",
        }
    }
)

print(type(restricted_hitl).__name__)


HumanInTheLoopMiddleware


## 7. 和本仓库 Harness 的对应关系

| LangChain HITL | 本仓库 Harness |
| --- | --- |
| `HumanInTheLoopMiddleware` | 工具风险审批逻辑 |
| `interrupts` | `approval_required` SSE 事件 |
| `action_requests` | 待审批工具调用信息 |
| `review_configs` | 前端可展示的审批动作 |
| `Command(resume=...)` | `stream_resume_approval(...)` |
| `thread_id` | approval ticket / session id / run id |
| checkpointer | 暂停任务的持久化状态 |

关键判断：

```text
HITL 不是弹一个确认框而已。
HITL 的本质是：任务可暂停、可审计、可恢复。
```

## 8. 本讲练习

请判断下面场景应该用 approve、reject 还是 edit：

1. 模型要写 `README.md`，内容正确，但路径应该改成 `docs/demo.md`
2. 模型要执行 `rm -rf /tmp/demo`
3. 模型要发送一封邮件，收件人和内容都正确

参考答案：

1. `edit`
2. `reject`
3. `approve`，但实际系统里仍然要记录审计信息

## 9. 本讲小结

这一讲你应该掌握：

- HITL 适合高风险工具调用
- 第一次调用返回 interrupt，不执行工具
- 恢复时必须使用同一个 `thread_id`
- `approve` 执行原工具调用
- `reject` 阻止工具调用
- `edit` 修改工具参数后执行
- 生产环境必须使用持久化 checkpointer

下一步可以继续学习 LangChain streaming，或者把这个 HITL 机制和本仓库现有审批页面做对照。